# BaCuSe synthesis parameter extraction

This notebook uses the package extraction functions for a local NOMAD export. When no archive export is present, it loads the supplied processed synthesis table. It does not make NOMAD network requests.

In [ ]:
import pandas as pd

from xrd_bo_optimization_tf.config import NOMAD_VOLUME_DIR, RESULTS_DATA_DIR, SYNTHESIS_CSV
from xrd_bo_optimization_tf.nomad_client import iter_archive_files
from xrd_bo_optimization_tf.synthesis import (
    build_synthesis_dataframe,
    build_synthesis_database,
)

archive_paths = list(iter_archive_files(NOMAD_VOLUME_DIR))
synthesis_db = build_synthesis_database(archive_paths) if archive_paths else {}
if synthesis_db:
    df = build_synthesis_dataframe(synthesis_db)
    data_source = f'{len(archive_paths)} local NOMAD archive files'
    extracted_output_path = RESULTS_DATA_DIR / 'bacuse2_synthesis_extraction.csv'
    extracted_output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(extracted_output_path, index=False)
    print(f'Wrote extracted synthesis table to {extracted_output_path}')
elif SYNTHESIS_CSV.is_file():
    df = pd.read_csv(SYNTHESIS_CSV)
    data_source = str(SYNTHESIS_CSV)
else:
    raise FileNotFoundError(
        f'No NOMAD archives found in {NOMAD_VOLUME_DIR} and no synthesis CSV at {SYNTHESIS_CSV}'
    )

print(f'Loaded {len(df)} synthesis records from {data_source}')
df.head()

In [ ]:
parameter_columns = [
    'spin_speed_rpm',
    'soft_bake_temp_c',
    'hard_bake_temp_c',
    'selenization_temp_c',
    'selenization_time_s',
]
available_columns = [column for column in parameter_columns if column in df]
completeness = df[available_columns].notna().sum().rename('available_samples').to_frame()
completeness['total_samples'] = len(df)
completeness

In [ ]:
from xrd_bo_optimization_tf.nomad_client import discover_bacuse_uploads

uploads = discover_bacuse_uploads(NOMAD_VOLUME_DIR)
if uploads:
    for upload_id, info in uploads.items():
        print(f'{upload_id}: {info["count"]} matching archives')
        print(f'  Samples: {info["sample_names"]}')
else:
    print(f'No local BaCuSe archives found under {NOMAD_VOLUME_DIR}')